## Standard Imports

In [3]:
import os
import re
import pandas as pd
from pathlib import Path
from urllib.parse import quote


print('Imported')

Imported


## Loading The Dataset And Creating Dataframe

In [21]:
csv_path = 'export_log.csv'

df = pd.read_csv(csv_path)
df.sample(5)

,description,region,composite_type,phase,start_date,end_date,scene_count,tiff_task_id,jpeg_task_id,tiff_drive_folder,jpeg_drive_folder,scale_m,crs,bands_tiff,bands_jpeg,export_timestamp
159,MON_Hasdeo_South_2023_06_first_event,Hasdeo_South,monthly,first_event,2023-06-01,2023-07-01,4,MAUBA26UDH3PW4NK55LDHZSN,RVZXWZ63WR2TCJVC45LXMXZA,Hasdeo_Dataset_300/GeoTIFF/monthly,Hasdeo_Dataset_300/JPEG/monthly,10,EPSG:32644,"B2,B3,B4,B8,B8A,B11,B12,SCL,NDVI,NDWI,NBR,EVI","B4,B3,B2 (RGB visualised)",2026-02-21T03:35:58.027661
147,MON_Hasdeo_South_2022_03_second_event,Hasdeo_South,monthly,second_event,2022-03-01,2022-04-01,25,7HZDCRRTS6UATRCLCL4RUS6A,7YHUVVIL7I6K2VXTTQO5INM5,Hasdeo_Dataset_300/GeoTIFF/monthly,Hasdeo_Dataset_300/JPEG/monthly,10,EPSG:32644,"B2,B3,B4,B8,B8A,B11,B12,SCL,NDVI,NDWI,NBR,EVI","B4,B3,B2 (RGB visualised)",2026-02-21T03:35:21.266076
57,MON_Hasdeo_North_2018_02_first_event,Hasdeo_North,monthly,first_event,2018-02-01,2018-03-01,1,IZETNFLJZJ2OOKXF6INUNR57,ASFB45H6OZIUXBZYAKOJHASM,Hasdeo_Dataset_300/GeoTIFF/monthly,Hasdeo_Dataset_300/JPEG/monthly,10,EPSG:32644,"B2,B3,B4,B8,B8A,B11,B12,SCL,NDVI,NDWI,NBR,EVI","B4,B3,B2 (RGB visualised)",2026-02-21T03:27:39.438203
68,MON_Hasdeo_North_2019_06_first_event,Hasdeo_North,monthly,first_event,2019-06-01,2019-07-01,4,EOTQ7HKCEIKSGNV75PTOPTXA,G56OJP62EDFDU2L25PD466VM,Hasdeo_Dataset_300/GeoTIFF/monthly,Hasdeo_Dataset_300/JPEG/monthly,10,EPSG:32644,"B2,B3,B4,B8,B8A,B11,B12,SCL,NDVI,NDWI,NBR,EVI","B4,B3,B2 (RGB visualised)",2026-02-21T03:30:46.940632
66,MON_Hasdeo_North_2019_04_first_event,Hasdeo_North,monthly,first_event,2019-04-01,2019-05-01,7,JNONZRS7SLYL42KKCHU2DX3O,I2DDWBLSBROI34MQNTQMFQDD,Hasdeo_Dataset_300/GeoTIFF/monthly,Hasdeo_Dataset_300/JPEG/monthly,10,EPSG:32644,"B2,B3,B4,B8,B8A,B11,B12,SCL,NDVI,NDWI,NBR,EVI","B4,B3,B2 (RGB visualised)",2026-02-21T03:30:42.623743


## CSV Info

In [24]:
df.shape

(268, 16)

In [26]:
df.info

<bound method DataFrame.info of                                 description       region composite_type  \
0       MON_Hasdeo_Full_2018_01_first_event  Hasdeo_Full        monthly   
1       MON_Hasdeo_Full_2018_02_first_event  Hasdeo_Full        monthly   
2       MON_Hasdeo_Full_2018_03_first_event  Hasdeo_Full        monthly   
3       MON_Hasdeo_Full_2018_04_first_event  Hasdeo_Full        monthly   
4       MON_Hasdeo_Full_2018_10_first_event  Hasdeo_Full        monthly   
..                                      ...          ...            ...   
263   SEAS_WET_Hasdeo_Core_2021_first_event  Hasdeo_Core   seasonal_wet   
264  SEAS_DRY_Hasdeo_Core_2022_second_event  Hasdeo_Core   seasonal_dry   
265   SEAS_WET_Hasdeo_Core_2022_first_event  Hasdeo_Core   seasonal_wet   
266   SEAS_DRY_Hasdeo_Core_2023_first_event  Hasdeo_Core   seasonal_dry   
267   SEAS_WET_Hasdeo_Core_2023_first_event  Hasdeo_Core   seasonal_wet   

            phase  start_date    end_date  scene_count  \
0     fir

In [27]:
df.describe

<bound method NDFrame.describe of                                 description       region composite_type  \
0       MON_Hasdeo_Full_2018_01_first_event  Hasdeo_Full        monthly   
1       MON_Hasdeo_Full_2018_02_first_event  Hasdeo_Full        monthly   
2       MON_Hasdeo_Full_2018_03_first_event  Hasdeo_Full        monthly   
3       MON_Hasdeo_Full_2018_04_first_event  Hasdeo_Full        monthly   
4       MON_Hasdeo_Full_2018_10_first_event  Hasdeo_Full        monthly   
..                                      ...          ...            ...   
263   SEAS_WET_Hasdeo_Core_2021_first_event  Hasdeo_Core   seasonal_wet   
264  SEAS_DRY_Hasdeo_Core_2022_second_event  Hasdeo_Core   seasonal_dry   
265   SEAS_WET_Hasdeo_Core_2022_first_event  Hasdeo_Core   seasonal_wet   
266   SEAS_DRY_Hasdeo_Core_2023_first_event  Hasdeo_Core   seasonal_dry   
267   SEAS_WET_Hasdeo_Core_2023_first_event  Hasdeo_Core   seasonal_wet   

            phase  start_date    end_date  scene_count  \
0     f

## Creating Directory Column For JPEG Format Images

## Creating Dataset From Scratch

Image Filename Parser & CSV Generator

Parses JPEG filenames in format: MON_Location_Direction_YYYY_MM_EventType.jpg
Maps local directory paths and constructs Google Drive TIFF links.

Usage:
    python generate_image_dataframe.py

Edit the CONFIG section below before running.

In [6]:
# ============================================================
#  CONFIG — Edit these values before running
# ============================================================

# Path to your local folder containing JPEG images
JPEG_FOLDER = r"tiff_to_jpeg"   

# Your Google Drive FOLDER link (shared folder containing TIFFs)
DRIVE_FOLDER_LINK = "https://drive.google.com/drive/folders/14MW_Qga2Y_h01NRbgxa2P9PJuD6E0sIj?usp=share_link" 

# Output CSV path
OUTPUT_CSV = "mata_data.csv"


In [7]:

# ============================================================
#  Filename pattern:
#  MON_Hasdeo_North_2022_05_slowdown.jpg
#  [0]  [1]     [2]   [3]  [4] [5]
# ============================================================

MONTH_MAP = {
    "JAN": "January",  "FEB": "February", "MAR": "March",
    "APR": "April",    "MAY": "May",       "JUN": "June",
    "JUL": "July",     "AUG": "August",    "SEP": "September",
    "OCT": "October",  "NOV": "November",  "DEC": "December",
    "MON": "Unknown"   # fallback if literal "MON" is used as placeholder
}


def parse_filename(filename: str) -> dict:
    """Parse a single image filename into structured fields."""
    stem = Path(filename).stem          # remove extension
    ext  = Path(filename).suffix.lower()
    parts = stem.split("_")

    result = {
        "filename":       filename,
        "file_stem":      stem,
        "extension":      ext,
        "month_code":     None,
        "month_name":     None,
        "location":       None,
        "direction":      None,
        "year":           None,
        "month_number":   None,
        "year_month":     None,
        "event_type":     None,
        "extra_fields":   None,
    }

    if len(parts) >= 6:
        result["month_code"]   = parts[0].upper()
        result["month_name"]   = MONTH_MAP.get(parts[0].upper(), parts[0])
        result["location"]     = parts[1]
        result["direction"]    = parts[2]
        result["year"]         = parts[3]
        result["month_number"] = parts[4]
        result["year_month"]   = f"{parts[3]}_{parts[4]}"
        result["event_type"]   = parts[5]
        if len(parts) > 6:
            result["extra_fields"] = "_".join(parts[6:])
    elif len(parts) == 5:
        result["month_code"]   = parts[0].upper()
        result["month_name"]   = MONTH_MAP.get(parts[0].upper(), parts[0])
        result["location"]     = parts[1]
        result["direction"]    = parts[2]
        result["year"]         = parts[3]
        result["event_type"]   = parts[4]
    else:
        # Best-effort: store whatever we have
        result["extra_fields"] = stem

    return result


def build_drive_search_link(folder_link: str, tiff_filename: str) -> str:
    """
    Build a Google Drive search link for the TIFF file inside the shared folder.
    Since Drive folder links can't deep-link to a specific file by name,
    we return the folder link + a note about the expected filename.
    If you have individual file IDs, replace this function.
    """
    return folder_link  # folder link — user can search by filename inside


def build_drive_direct_link(file_id: str) -> str:
    """Helper if you have individual file IDs."""
    return f"https://drive.google.com/file/d/{file_id}/view"


def scan_folder(folder_path: str) -> pd.DataFrame:
    """Scan folder for JPEG images, parse filenames, return DataFrame."""
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    # Collect all JPEG files
    jpeg_extensions = {".jpg", ".jpeg"}
    image_files = [
        f for f in folder.iterdir()
        if f.is_file() and f.suffix.lower() in jpeg_extensions
    ]

    if not image_files:
        print(f"No JPEG files found in: {folder_path}")
        return pd.DataFrame()

    print(f"✅ Found {len(image_files)} JPEG file(s) in: {folder_path}")

    rows = []
    for img_path in sorted(image_files):
        parsed = parse_filename(img_path.name)

        # Construct expected TIFF filename (same stem, .tif extension)
        tiff_filename = img_path.stem + ".tif"
        tiff_filename_alt = img_path.stem + ".tiff"

        row = {
            # --- File Info ---
            "jpeg_filename":        img_path.name,
            "tiff_filename":        tiff_filename,

            # --- Parsed Keywords ---
            "month_code":           parsed["month_code"],
            "month_name":           parsed["month_name"],
            "location":             parsed["location"],
            "direction":            parsed["direction"],
            "year":                 parsed["year"],
            "month_number":         parsed["month_number"],
            "year_month":           parsed["year_month"],
            "event_type":           parsed["event_type"],
            "extra_fields":         parsed["extra_fields"],

            # --- Paths ---
            "local_jpeg_directory": str(folder.resolve()),
            "local_jpeg_full_path": str(img_path.resolve()),

            # --- Google Drive ---
            "drive_folder_link":    DRIVE_FOLDER_LINK,
            "expected_tiff_name":   tiff_filename,
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    return df


def main():
    print("=" * 60)
    print("  Image Filename Parser & CSV Generator")
    print("=" * 60)

    # --- Scan folder ---
    df = scan_folder(JPEG_FOLDER)

    if df.empty:
        print("No data to save. Exiting.")
        return

    # --- Save CSV ---
    output_path = Path(OUTPUT_CSV)
    df.to_csv(output_path, index=False)
    print(f"\n✅ CSV saved to: {output_path.resolve()}")
    print(f"   Rows: {len(df)}  |  Columns: {len(df.columns)}")
    print(f"\n📋 Columns in CSV:")
    for col in df.columns:
        print(f"   • {col}")

    print("\n📊 Preview (first 3 rows):")
    print(df.head(3).to_string(index=False))


# ============================================================
#  BONUS: If you have a Drive file ID mapping CSV/dict,
#  use this function to merge TIFF file IDs into the dataframe.
# ============================================================
def merge_drive_file_ids(df: pd.DataFrame, id_mapping: dict) -> pd.DataFrame:
    """
    id_mapping: dict like {"MON_Hasdeo_North_2022_05_slowdown.tif": "1AbcDriveFileID"}
    Adds a 'drive_direct_link' column with per-file links.
    """
    df["drive_file_id"] = df["tiff_filename"].map(id_mapping)
    df["drive_direct_link"] = df["drive_file_id"].apply(
        lambda fid: build_drive_direct_link(fid) if pd.notna(fid) else None
    )
    return df


if __name__ == "__main__":
    main()

  Image Filename Parser & CSV Generator
✅ Found 374 JPEG file(s) in: tiff_to_jpeg

✅ CSV saved to: /Users/manjushwarkhairkar/GitHub/Urban-Leaf-Health-Monitoring/data/02_comparison_based_on_events/mata_data.csv
   Rows: 374  |  Columns: 15

📋 Columns in CSV:
   • jpeg_filename
   • tiff_filename
   • month_code
   • month_name
   • location
   • direction
   • year
   • month_number
   • year_month
   • event_type
   • extra_fields
   • local_jpeg_directory
   • local_jpeg_full_path
   • drive_folder_link
   • expected_tiff_name

📊 Preview (first 3 rows):
              jpeg_filename               tiff_filename month_code month_name location direction year month_number year_month event_type            extra_fields                                                                                          local_jpeg_directory                                                                                                                      local_jpeg_full_path                               

## Verification

In [10]:

csv_path = r'mata_data.csv'
df = pd.read_csv(csv_path)

df.sample(5)

,jpeg_filename,tiff_filename,month_code,month_name,location,direction,year,month_number,year_month,event_type,extra_fields,local_jpeg_directory,local_jpeg_full_path,drive_folder_link,expected_tiff_name
358,MON_Mining_Buffer_2023_12_renewed.jpg,MON_Mining_Buffer_2023_12_renewed.tif,MON,Unknown,Mining,Buffer,2023.0,12.0,2023_12,renewed,NaN,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,https://drive.google.com/drive/folders/14MW_Qg...,MON_Mining_Buffer_2023_12_renewed.tif
270,MON_Kente_Extension_2022_03_slowdown.jpg,MON_Kente_Extension_2022_03_slowdown.tif,MON,Unknown,Kente,Extension,2022.0,3.0,2022_03,slowdown,NaN,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,https://drive.google.com/drive/folders/14MW_Qg...,MON_Kente_Extension_2022_03_slowdown.tif
373,MON_Mining_Buffer_2025_06_renewed.jpg,MON_Mining_Buffer_2025_06_renewed.tif,MON,Unknown,Mining,Buffer,2025.0,6.0,2025_06,renewed,NaN,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,https://drive.google.com/drive/folders/14MW_Qg...,MON_Mining_Buffer_2025_06_renewed.tif
234,MON_Hasdeo_South_2023_10_renewed.jpg,MON_Hasdeo_South_2023_10_renewed.tif,MON,Unknown,Hasdeo,South,2023.0,10.0,2023_10,renewed,NaN,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,https://drive.google.com/drive/folders/14MW_Qg...,MON_Hasdeo_South_2023_10_renewed.tif
197,MON_Hasdeo_South_2019_06_rapid_exp.jpg,MON_Hasdeo_South_2019_06_rapid_exp.tif,MON,Unknown,Hasdeo,South,2019.0,6.0,2019_06,rapid,exp,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,/Users/manjushwarkhairkar/GitHub/Urban-Leaf-He...,https://drive.google.com/drive/folders/14MW_Qg...,MON_Hasdeo_South_2019_06_rapid_exp.tif
